# Configs — Upfront Workflow + Waterfall Resolution

This notebook demonstrates the configuration lifecycle across the four scopes:
**code default → platform → catalog → collection**.  Every `PluginConfig` is
resolved through this waterfall; services never branch on `cfg is None`.

## Scenarios

1. **Upfront creation** — `PUT .../collections/{col}/configs/{key}` JIT-creates the collection if it does not exist.
2. **Waterfall** — platform < catalog < collection overrides; the effective config reports the source of each field.
3. **Bulk apply** — several configs in one call.
4. **Discovery** — enumerate registered schemas and drivers.
5. **Default resolution** — `DELETE` a collection config, the waterfall falls back to catalog/platform/code defaults.

In [ ]:
import os
import httpx

BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"
CATALOG_ID = "demo-waterfall"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

r = await client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "Waterfall Demo",
    "description": "Catalog for waterfall + upfront-config demonstrations.",
})
_check(r, "Catalog")

---
## 1. Upfront config creation — JIT collection

Put a `CollectionSchema` onto a collection that **does not exist yet**.  The
service calls `ensure_collection_exists(...)` transparently: the thin registry
row is created with waterfall-resolved defaults for every sub-config, and the
config is stored.  No 404, no 409, no ordering constraint on the client.

In [ ]:
COLL_UPFRONT = "ghost-collection"

# Collection does NOT exist yet — PUT still succeeds.
schema = {
    "fields": [
        {"name": "temperature", "data_type": "double", "required": False},
        {"name": "station_id",  "data_type": "string", "required": True},
    ],
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_UPFRONT}/configs/collection:schema",
    json=schema,
)
_check(r, "Upfront PUT CollectionSchema (before collection existed)")

# Verify the collection was JIT-created
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPFRONT}")
_check(r, "GET collection after upfront PUT")
print("  id =", r.json().get("id"))

---
## 2. Waterfall — platform < catalog < collection

Set `CollectionRoutingConfig` at three levels and ask the effective endpoint
to report which scope contributed each field.  The resolution order is:

```
code default  →  platform  →  catalog  →  collection
```

`ConfigResolutionError` (HTTP 500) is raised only if the waterfall produces
no usable value at any scope — this signals an ops/bootstrap bug, never a
user-action problem.

In [ ]:
COLL_WATERFALL = "routing-demo"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_WATERFALL,
    "title": "Routing Waterfall",
    "description": "Collection used to demonstrate three-tier routing waterfall.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

# Platform default — applies to every catalog unless overridden
r = await client.put(
    "/configs/platform/configs/collection:routing",
    json={"operations": {"WRITE": ["CollectionPostgresqlDriver"], "READ": ["CollectionPostgresqlDriver"]}},
)
_check(r, "Platform routing")

# Catalog override — applies to every collection inside CATALOG_ID
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/configs/collection:routing",
    json={"operations": {"WRITE": ["CollectionPostgresqlDriver"], "READ": ["CollectionElasticsearchDriver", "CollectionPostgresqlDriver"]}},
)
_check(r, "Catalog routing (ES read preferred)")

# Collection override — applies only to COLL_WATERFALL
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_WATERFALL}/configs/collection:routing",
    json={"operations": {"WRITE": ["CollectionPostgresqlDriver", "CollectionElasticsearchDriver"]}},
)
_check(r, "Collection routing (dual WRITE)")

In [ ]:
# Effective view — which scope contributed each field
r = await client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_WATERFALL}/configs/collection:routing/effective",
)
_check(r, "Effective config")
print(r.json())

---
## 3. Bulk apply — several configs in one call

Send schema + routing + driver config together. The endpoint JIT-creates the
collection if necessary (same path as Scenario 1).

In [ ]:
COLL_BULK = "bulk-configured"

bulk = {
    "collection:schema": {
        "fields": [
            {"name": "name", "data_type": "string", "required": True},
            {"name": "value", "data_type": "double"},
        ],
    },
    "collection:routing": {
        "operations": {"WRITE": ["CollectionPostgresqlDriver"], "READ": ["CollectionPostgresqlDriver"]},
    },
    "collection:write_policy": {
        "on_conflict": "update",
        "external_id_field": "properties.name",
    },
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_BULK}/bulk",
    json=bulk,
)
_check(r, "Bulk apply (JIT collection + 3 configs)")

---
## 4. Discovery — registered schemas and drivers

What `PluginConfig` classes are registered? What storage drivers does this
deploy expose? Use the discovery endpoints to answer without touching source.

In [ ]:
r = await client.get("/configs/schemas")
_check(r, "Schemas")
keys = sorted(r.json().get("keys", [])) if r.status_code < 400 else []
print(f"  {len(keys)} registered config classes")
for k in keys[:12]:
    print("   ", k)
if len(keys) > 12:
    print("    ...")

# Inspect one specific schema
if keys:
    key = "collection:routing" if "collection:routing" in keys else keys[0]
    r = await client.get(f"/configs/schemas/{key}")
    _check(r, f"Schema {key}")

r = await client.get("/configs/storage/drivers")
_check(r, "Storage drivers")
drivers = r.json() if r.status_code < 400 else []
print(f"  {len(drivers)} drivers available")

---
## 5. Default resolution — DELETE collection config, waterfall back-fills

Remove the collection-scope `CollectionRoutingConfig`. The collection keeps
serving traffic because the catalog override (then platform, then code default)
back-fills the missing fields.  **No 404, no 500.**

In [ ]:
r = await client.delete(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_WATERFALL}/configs/collection:routing",
)
_check(r, "DELETE collection-scope routing")

# Effective resolution must still succeed; source of each field now shifts
# down a tier (catalog → platform → code default).
r = await client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_WATERFALL}/configs/collection:routing/effective",
)
_check(r, "Effective after DELETE")
print(r.json())

In [ ]:
# Revert platform-scope change so later notebooks start from a clean slate
r = await client.delete("/configs/platform/configs/collection:routing")
_check(r, "Revert platform routing")

r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup catalog")
await client.aclose()